# 05 · Data for the website

This notebook prepares the data for the website in the `docs/` folder. GitHub Pages can publish that folder directly.

The website can't read 2.7 million rows quickly, so this notebook counts everything once and saves one small file: `docs/data/dashboard.json`. The website then computes the shares and rankings in the browser.

## What is counted

Only complete months and vacancies with at least one skill are used, as in notebooks 03 and 04.

For every occupation domain (plus "All occupations") and every year:

- the number of vacancies
- the number of vacancies that ask for at least one skill of each group
- the number of vacancies that ask for each skill

The website divides these to get **vacancy shares**: the percentage of vacancies that mention a skill or group. Raw counts are not shown as trends, because ads list more skills every year.

## Inputs

- `data/processed/vacancies` and `vacancy_skills` from notebook 02
- `data/processed/skill_categories` from notebook 03

Run notebooks 01 to 03 first. Re-run this notebook every time you change the overrides file.

## 0. Settings and loading

In [ ]:
from pathlib import Path
from datetime import date
import json
import pandas as pd
from IPython.display import display

pd.set_option('display.width', 200)

# The same periods and thresholds as notebook 04. The website uses them for the
# "emerging / declining" labels.
EARLY_YEARS = [2017, 2018, 2019]
RECENT_YEARS = [2023, 2024, 2025]
CHANGE_FACTOR = 1.5
MIN_VACANCIES = 200          # for "All occupations"
MIN_VACANCIES_DOMAIN = 50    # for one occupation domain

# Inside one domain, keep the top skills of each group in the early and in the
# recent period. "All occupations" keeps every skill, so the search box finds any skill.
TOP_PER_GROUP_DOMAIN = 60

PARTIAL_YEAR = 2026
PARTIAL_NOTE = 'January to August only'

GROUPS = ['Languages', 'Soft skills', 'Basic digital', 'Technical / IT',
          'Knowledge', 'Job-specific skills']

DOMAIN_NAMES = {
    'A': 'Agriculture and nature', 'B': 'Arts and crafts',
    'C': 'Banking, insurance and real estate', 'D': 'Retail and sales',
    'E': 'Communication and media', 'F': 'Construction',
    'G': 'Hospitality and tourism', 'H': 'Industry',
    'I': 'Installation and maintenance', 'J': 'Health',
    'K': 'Personal and community services', 'L': 'Performing arts',
    'M': 'Business support (admin, finance, IT, HR)', 'N': 'Transport and logistics',
}

# Support running from either the project root or notebooks/.
project_dir = Path.cwd()
if not (project_dir / 'data' / 'processed').is_dir():
    project_dir = project_dir.parent
processed_dir = project_dir / 'data' / 'processed'
site_data_dir = project_dir / 'docs' / 'data'
site_data_dir.mkdir(parents=True, exist_ok=True)

def load_table(name):
    parquet = processed_dir / f'{name}.parquet'
    if parquet.exists():
        return pd.read_parquet(parquet)
    return pd.read_csv(processed_dir / f'{name}.csv.gz')

vacancies = load_table('vacancies')
vacancy_skills = load_table('vacancy_skills')
skill_categories = load_table('skill_categories')
print('vacancies:', f'{len(vacancies):,}', '| vacancy_skills:', f'{len(vacancy_skills):,}',
      '| skill_categories:', f'{len(skill_categories):,}')

unknown_groups = set(skill_categories['group']) - set(GROUPS)
assert not unknown_groups, f'Unknown groups in skill_categories: {unknown_groups}'

## 1. Vacancies used

Complete months only, and only vacancies with at least one skill. The occupation domain is the first letter of the ROME code.

In [ ]:
vac = vacancies[vacancies['in_complete_period'] & vacancies['has_skills']].copy()
vac['domain'] = vac['occupation_code'].astype(str).str[0]
vac = vac[['vacancy_key', 'year', 'domain']]
YEARS = sorted(vac['year'].unique().tolist())
print('years:', YEARS[0], 'to', YEARS[-1], '| vacancies used:', f'{len(vac):,}')

unknown_domains = sorted(set(vac['domain']) - set(DOMAIN_NAMES))
print('domains not in DOMAIN_NAMES (left out of the domain filter):', unknown_domains or 'none')

per_domain_year = pd.crosstab(vac['domain'], vac['year'])
display(per_domain_year)

## 2. Counts per group and per skill

One row of `pairs` = one vacancy asking for one skill, with the vacancy's year and domain and the skill's group.

In [ ]:
cats = skill_categories[['skill_uri', 'skill', 'group', 'subgroup']].copy()
cats['skill_uri'] = cats['skill_uri'].astype(str)

pairs = (vacancy_skills[['vacancy_key', 'skill_uri']]
         .assign(skill_uri=lambda d: d['skill_uri'].astype(str))
         .drop_duplicates()
         .merge(vac, on='vacancy_key')
         .merge(cats[['skill_uri', 'group']], on='skill_uri', how='left'))
print('vacancy-skill pairs used:', f'{len(pairs):,}')
assert pairs['group'].notna().all(), 'Some skills have no group. Run notebook 03 again.'

def yearly(counts):
    # counts: Series indexed by year -> list with one number per year in YEARS
    return [int(counts.get(y, 0)) for y in YEARS]

# Vacancies per domain and year ('ALL' = all occupations).
vacancy_counts = {'ALL': yearly(vac.groupby('year').size())}
for d in DOMAIN_NAMES:
    vacancy_counts[d] = yearly(vac[vac['domain'] == d].groupby('year').size())

# Vacancies asking for at least one skill of each group.
vac_group = pairs.drop_duplicates(['vacancy_key', 'group'])
group_counts = {'ALL': [yearly(vac_group[vac_group['group'] == g].groupby('year').size()) for g in GROUPS]}
for d in DOMAIN_NAMES:
    part = vac_group[vac_group['domain'] == d]
    group_counts[d] = [yearly(part[part['group'] == g].groupby('year').size()) for g in GROUPS]

# Vacancies asking for each skill, per year: all occupations, then per domain.
skill_year_all = pairs.groupby(['skill_uri', 'year']).size().unstack(fill_value=0)
skill_year_dom = pairs.groupby(['domain', 'skill_uri', 'year']).size().unstack(fill_value=0)
print('skills x years table:', skill_year_all.shape)

## 3. Which skills go into the file

- **All occupations**: every skill, so the search box on the website can find any of them.
- **One domain**: for each group, the top skills of the early period and the top skills of the recent period (so declining skills are kept too). This keeps the file small.

In [ ]:
skill_list = cats.merge(skill_year_all.sum(axis=1).rename('total').reset_index(), on='skill_uri', how='left')
skill_list['total'] = skill_list['total'].fillna(0)
skill_list = skill_list.sort_values('total', ascending=False, ignore_index=True)
skill_id = {uri: i for i, uri in enumerate(skill_list['skill_uri'])}
group_of = dict(zip(skill_list['skill_uri'], skill_list['group']))

def series_dict(table):
    out = {}
    for uri, row in table.iterrows():
        values = yearly(row)
        if sum(values) > 0:
            out[str(skill_id[uri])] = values
    return out

skill_counts = {'ALL': series_dict(skill_year_all)}
early_cols = [y for y in EARLY_YEARS if y in skill_year_dom.columns]
recent_cols = [y for y in RECENT_YEARS if y in skill_year_dom.columns]
for d in DOMAIN_NAMES:
    if d not in skill_year_dom.index.get_level_values(0):
        skill_counts[d] = {}
        continue
    table = skill_year_dom.loc[d]
    keep = set()
    for g in GROUPS:
        in_group = table[table.index.map(group_of) == g]
        for cols in [early_cols, recent_cols]:
            keep |= set(in_group[cols].sum(axis=1).nlargest(TOP_PER_GROUP_DOMAIN).index)
    skill_counts[d] = series_dict(table.loc[sorted(keep)])

summary = pd.DataFrame({'skills kept': {k: len(v) for k, v in skill_counts.items()}})
summary['domain'] = summary.index.map(lambda k: 'All occupations' if k == 'ALL' else DOMAIN_NAMES[k])
display(summary)

## 4. Save `docs/data/dashboard.json`

In [ ]:
dashboard = {
    'meta': {
        'generated': date.today().isoformat(),
        'source': 'ADEM skills-vacancies open data, skills classified with ESCO v1.2.1',
        'years': YEARS,
        'partial_year': PARTIAL_YEAR,
        'partial_note': PARTIAL_NOTE,
        'early_years': EARLY_YEARS,
        'recent_years': RECENT_YEARS,
        'change_factor': CHANGE_FACTOR,
        'min_vacancies': MIN_VACANCIES,
        'min_vacancies_domain': MIN_VACANCIES_DOMAIN,
    },
    'groups': GROUPS,
    'domains': [{'code': 'ALL', 'name': 'All occupations'}]
               + [{'code': k, 'name': v} for k, v in DOMAIN_NAMES.items()],
    # One entry per skill: [name, group number, subgroup]. Its position is its id.
    'skills': [[s, GROUPS.index(g), sg] for s, g, sg in
               zip(skill_list['skill'], skill_list['group'], skill_list['subgroup'].fillna('other'))],
    'vacancies': vacancy_counts,
    'group_counts': group_counts,
    'skill_counts': skill_counts,
}

out_path = site_data_dir / 'dashboard.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(dashboard, f, ensure_ascii=False, separators=(',', ':'))
print('Saved:', out_path, f'({out_path.stat().st_size / 1e6:.2f} MB)')

## 5. Checks

The shares computed from the file must match notebook 03 (for example soft skills in 2025: 76.8% of vacancies).

In [ ]:
check = json.load(open(out_path, encoding='utf-8'))
all_vac = check['vacancies']['ALL']
group_share = pd.DataFrame({g: [100 * n / v for n, v in zip(check['group_counts']['ALL'][i], all_vac)]
                            for i, g in enumerate(GROUPS)}, index=YEARS).round(1)
display(group_share)

no_share_above_100 = all(n <= v for d in check['vacancies']
                         for series in check['group_counts'][d]
                         for n, v in zip(series, check['vacancies'][d]))
domains_add_up = all(sum(check['vacancies'][d][i] for d in DOMAIN_NAMES) <= all_vac[i]
                     for i in range(len(YEARS)))
checks = {
    'every skill is in the file': len(check['skills']) == len(skill_categories),
    'no group share above 100%': no_share_above_100,
    'domain vacancies do not exceed the total': domains_add_up,
    'total vacancies match': sum(all_vac) == len(vac),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)

## 6. Look at the website on your Mac

The website reads the JSON file, so it needs a small local web server (opening `index.html` by double-click won't load the data). In a terminal:

```
cd "/Users/andrecorreia/Desktop/The Projet/luxembourg-job-market/docs"
python3 -m http.server 8000
```

Then open http://localhost:8000 in your browser. Press Ctrl+C in the terminal to stop the server.

## Publish on GitHub Pages

1. Push the project to GitHub. The `docs/` folder must be included.
2. On GitHub, open the repository, then **Settings → Pages**.
3. Under "Build and deployment", choose **Deploy from a branch**, branch `main`, folder `/docs`, then **Save**.
4. After a minute or two the site is live at `https://<your-username>.github.io/<repository-name>/`.

After changing the overrides, run notebooks 03 and 05 again, then commit and push `docs/data/dashboard.json`.